# AfroViewz 01: Data acquisition

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JBWandera/Capstone/blob/afroviewz/notebooks/01_data_acquisition.ipynb)

**Workflow stage:** data acquisition (CRISP-DM: data understanding).

**Goal:** collect every image source, convert each one to a single format (COCO, one class: `pothole`), and keep a cleaned, downscaled copy on Google Drive. Nothing is split or merged here; that happens in notebook 02.

| Source | Region | Role | License |
|---|---|---|---|
| Stellenbosch Road Pothole Images | South Africa | train (incl. pothole free frames as hard negatives) | CC0 1.0 |
| Roboflow `pothole-detection-6vpbs` | Mixed | train (polygon masks for severity) | CC BY 4.0 (confirm) |
| Roboflow `potholes-detection-jbnou` (the v1 capstone data) | Mixed | train, and comparison with v1 | CC BY 4.0 (confirm) |
| Mapillary street photos, Uganda | Kampala, Entebbe, Masaka, Mbale, Lira | **holdout**: never trained on | CC BY-SA 4.0 |
| Own phone clips (optional) | Uganda | holdout | own |

The source list lives in `configs/sources.yaml`, so this notebook holds no hard coded paths or keys.

**Runtime:** CPU is enough. The Stellenbosch download is about 23 GB and takes the longest; it goes to the Colab local disk and only the downscaled copy (max side 1280 px) is written to Drive.

## 1. Setup

In [ ]:
# Run this first. Mounts Drive (data is kept there) and installs the afroviewz package.
from google.colab import drive
drive.mount("/content/drive")

!git clone -q -b afroviewz https://github.com/JBWandera/Capstone.git /content/Capstone 2>/dev/null || git -C /content/Capstone pull -q
%cd /content/Capstone
!pip install -q -e ".[data]"


## 2. Secrets

Add these in the Colab key icon (Secrets) panel, and switch on notebook access:

* `ROBOFLOW_API_KEY`: Roboflow, Settings, API Keys. **Use a new key**: the old one was published in the v1 notebook and must be revoked.
* `MAPILLARY_TOKEN`: create a free app at https://www.mapillary.com/dashboard/developers and copy the *client access token*.

Keys are read here and never printed or saved to the notebook.

In [ ]:
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        print(f"Secret {name} not set")
        return None

ROBOFLOW_API_KEY = secret("ROBOFLOW_API_KEY")
MAPILLARY_TOKEN = secret("MAPILLARY_TOKEN")


## 3. The source register

In [ ]:
import pandas as pd
from afroviewz import paths
from afroviewz.data import sources, build, coco as C

cfg = sources.load_config()
print("raw downloads ->", paths.raw_root())
print("cleaned copies ->", paths.data_root() / "interim")
pd.DataFrame(cfg).T[["title", "region", "role", "license", "download"]]


## 4. Download

Each source is downloaded only if its raw folder is empty, so the cell can be re-run after a disconnect.

In [ ]:
def fetch(name):
    d = paths.raw_dir(name)
    if d.exists() and any(d.iterdir()):
        print(f"{name}: already downloaded at {d}")
        return d
    return sources.download(name, cfg, roboflow_key=ROBOFLOW_API_KEY, mapillary_token=MAPILLARY_TOKEN)

for name in ["capstone_v1", "roboflow_seg"]:
    fetch(name)


In [ ]:
# About 23 GB, allow 20 to 40 minutes.
fetch("stellenbosch")


In [ ]:
# Ugandan street photos. Prints how many images each town has before and after thinning.
fetch("uganda_mapillary")


## 5. Convert, clean and store each source

For every source: map all pothole-like categories to one class and drop the rest, clip boxes to the image, drop boxes under 4 px and exact duplicate boxes, then save a JPEG copy with the long side at 1280 px and the boxes scaled to match. Each source gets a `report.json` with before and after counts.

In [ ]:
reports = {}
for name in ["capstone_v1", "roboflow_seg", "stellenbosch", "uganda_mapillary"]:
    out = paths.interim_dir(name)
    if (out / "coco.json").exists():
        print(f"{name}: already prepared")
        reports[name] = __import__("json").loads((out / "report.json").read_text())
        continue
    coco, root = sources.to_coco(name, cfg)
    reports[name] = build.prepare_source(name, coco, root, out)

rows = []
for name, r in reports.items():
    rows.append({
        "source": name,
        "images_in": r["before"]["images"],
        "boxes_in": r["before"]["annotations"],
        "images_out": r["after"]["images"],
        "boxes_out": r["after"]["annotations"],
        "with_potholes": r["after"]["images_with_potholes"],
        "pothole_free": r["after"]["negative_images"],
        "boxes_clipped": r["boxes"]["clipped"],
        "tiny_dropped": r["boxes"]["dropped_tiny"],
        "dup_boxes_dropped": r["boxes"]["dropped_duplicate"],
        "other_classes_dropped": sum(r["categories"]["dropped_by_category"].values()),
        "unreadable_images": r["image_problem_count"],
    })
acq = pd.DataFrame(rows).set_index("source")
acq.to_csv(paths.data_root() / "interim" / "acquisition_summary.csv")
acq


## 6. A first look

A few random images per source with their boxes, to confirm the conversion is right before any modelling. The Uganda Mapillary photos have no boxes yet: they are labelled in notebook 04.

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def show_samples(name, n=6, seed=0):
    d = paths.interim_dir(name)
    coco = C.load(d / "coco.json")
    by_img = {}
    for a in coco["annotations"]:
        by_img.setdefault(a["image_id"], []).append(a["bbox"])
    pool = [im for im in coco["images"] if im["id"] in by_img] or coco["images"]
    picks = random.Random(seed).sample(pool, min(n, len(pool)))
    fig, axes = plt.subplots(1, len(picks), figsize=(4 * len(picks), 3.4))
    for ax, im in zip(axes, picks):
        ax.imshow(Image.open(d / "images" / im["file_name"]))
        for x, y, w, h in by_img.get(im["id"], []):
            ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, lw=2, ec="#ff7a00"))
        ax.set_axis_off()
    fig.suptitle(name)
    plt.tight_layout()
    plt.show()

for name in reports:
    show_samples(name)


## 7. Uganda coverage map

Where the Ugandan holdout photos were taken. This is also the first view of what the app map will show.

In [ ]:
meta = pd.read_csv(paths.raw_dir("uganda_mapillary") / "metadata.csv")
print(meta.groupby("area").size().rename("images"))
fig, ax = plt.subplots(figsize=(6, 6))
for area, g in meta.groupby("area"):
    ax.scatter(g.lon, g.lat, s=4, label=area)
ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); ax.legend(); ax.set_title("Mapillary holdout images")
plt.show()
meta.to_csv(paths.interim_dir("uganda_mapillary") / "metadata.csv", index=False)


## Outputs

On Drive under `AfroViewz/data/interim/<source>/`: `images/`, `coco.json`, `report.json`, plus `acquisition_summary.csv`.

**Next:** notebook 02 merges the sources, finds near duplicates and makes a leak free train, valid, test split.